# 🚨 KPMG Data Engineer Interview Experience

📌 **Role:** Data Engineer

## Round 1 – SQL + Spark + Data Processing

1️⃣ Solve complex SQL problems using **window functions, CTEs, and deduplication logic**

2️⃣ Handle **Slowly Changing Dimensions (SCD Type 1 & 2)** with examples

3️⃣ Optimize queries — **indexing, partition pruning, and join strategies**

4️⃣ Write queries for **data validation & reconciliation (source vs target)**

5️⃣ **Scenario:** Detect anomalies in transaction data using SQL



In [0]:
# 1. Solve complex SQL problems using window functions, CTEs, and deduplication logic

WITH ranked_data AS (
    SELECT *,
           ROW_NUMBER() OVER (
               PARTITION BY customer_id
               ORDER BY updated_at DESC
           ) AS rn
    FROM customers
)
SELECT *
FROM ranked_data
WHERE rn = 1;

2. Handle Slowly Changing Dimensions (SCD Type 1 & 2) with examples

## SCD Type 1 & Type 2 Using PySpark

Assume we have a **customer dimension** and incoming customer data.

### Sample Data

```python
from pyspark.sql import functions as F
from pyspark.sql.window import Window

customer_dim = spark.createDataFrame([
    (101, "Rahul", "Kolkata"),
    (102, "Amit", "Delhi")
], ["customer_id", "name", "city"])

customer_dim.show()
```

Incoming data:

```python
source_df = spark.createDataFrame([
    (101, "Rahul", "Bangalore"),
    (103, "Priya", "Mumbai")
], ["customer_id", "name", "city"])
```

---

# 🔹 SCD Type 1 — Overwrite

In Type 1, we **replace the old value with the new value**.

```python
# Update existing customers
updated_df = (
    customer_dim.alias("target")
    .join(
        source_df.alias("source"),
        "customer_id",
        "full"
    )
    .select(
        F.coalesce(
            F.col("source.customer_id"),
            F.col("target.customer_id")
        ).alias("customer_id"),

        F.coalesce(
            F.col("source.name"),
            F.col("target.name")
        ).alias("name"),

        F.coalesce(
            F.col("source.city"),
            F.col("target.city")
        ).alias("city")
    )
)

updated_df.show()
```

### Result

```text
+-----------+-----+---------+
|customer_id|name |city     |
+-----------+-----+---------+
|101        |Rahul|Bangalore|
|102        |Amit |Delhi    |
|103        |Priya|Mumbai   |
+-----------+-----+---------+
```

👉 Customer `101`'s old city **Kolkata** is replaced by **Bangalore**.

---

# 🔹 SCD Type 2 — Maintain History

For Type 2, we maintain:

* `start_date`
* `end_date`
* `is_current`

### Existing Dimension

```python
customer_dim = spark.createDataFrame([
    (101, "Rahul", "Kolkata", "2026-01-01", None, 1),
    (102, "Amit", "Delhi", "2026-01-01", None, 1)
], [
    "customer_id",
    "name",
    "city",
    "start_date",
    "end_date",
    "is_current"
])
```

Convert date columns:

```python
customer_dim = (
    customer_dim
    .withColumn("start_date", F.to_date("start_date"))
    .withColumn("end_date", F.to_date("end_date"))
)
```

Incoming data:

```python
source_df = spark.createDataFrame([
    (101, "Rahul", "Bangalore"),
    (103, "Priya", "Mumbai")
], ["customer_id", "name", "city"])
```

---

## Step 1 — Identify Changed Records

```python
changed_records = (
    customer_dim.alias("target")
    .join(
        source_df.alias("source"),
        "customer_id"
    )
    .filter(
        (F.col("target.is_current") == 1) &
        (
            (F.col("target.name") != F.col("source.name")) |
            (F.col("target.city") != F.col("source.city"))
        )
    )
    .select("customer_id")
)

changed_records.show()
```

Customer `101` will be identified as a changed record.

---

## Step 2 — Expire the Old Record

```python
expired_df = (
    customer_dim.alias("target")
    .join(
        changed_records.alias("changed"),
        "customer_id",
        "left"
    )
    .withColumn(
        "end_date",
        F.when(
            (F.col("changed.customer_id").isNotNull()) &
            (F.col("target.is_current") == 1),
            F.date_sub(F.current_date(), 1)
        ).otherwise(F.col("target.end_date"))
    )
    .withColumn(
        "is_current",
        F.when(
            F.col("changed.customer_id").isNotNull() &
            (F.col("target.is_current") == 1),
            F.lit(0)
        ).otherwise(F.col("target.is_current"))
    )
    .select(
        "customer_id",
        "name",
        "city",
        "start_date",
        "end_date",
        "is_current"
    )
)
```

---

## Step 3 — Create New Version

```python
new_records = (
    source_df.alias("source")
    .join(
        customer_dim.alias("target"),
        "customer_id",
        "left"
    )
    .filter(
        F.col("target.customer_id").isNull() |
        (
            (F.col("target.is_current") == 1) &
            (
                (F.col("target.name") != F.col("source.name")) |
                (F.col("target.city") != F.col("source.city"))
            )
        )
    )
    .select(
        F.col("source.customer_id"),
        F.col("source.name"),
        F.col("source.city"),
        F.current_date().alias("start_date"),
        F.lit(None).cast("date").alias("end_date"),
        F.lit(1).alias("is_current")
    )
)
```

---

## Step 4 — Combine Old + New Records

```python
final_df = expired_df.unionByName(new_records)

final_df.orderBy("customer_id", "start_date").show()
```

### Expected Result

```text
+-----------+-----+---------+----------+----------+----------+
|customer_id|name |city     |start_date|end_date  |is_current|
+-----------+-----+---------+----------+----------+----------+
|101        |Rahul|Kolkata  |2026-01-01|2026-08-31|0         |
|101        |Rahul|Bangalore|2026-09-01|null      |1         |
|102        |Amit |Delhi    |2026-01-01|null      |1         |
|103        |Priya|Mumbai   |2026-09-01|null      |1         |
+-----------+-----+---------+----------+----------+----------+
```

### ⭐ Interview Summary

**SCD Type 1:**

```text
Source → Compare → Update existing record
                     ↓
               No history
```

**SCD Type 2:**

```text
Source
   ↓
Compare with Target
   ↓
Changed?
 ┌───────┴───────┐
Yes              No
 ↓                ↓
Expire old      Keep old
record
 ↓
Insert new version
 ↓
Maintain History
```

> **In a real Databricks project, SCD Type 2 is commonly implemented using Delta Lake `MERGE`, which makes the update-and-insert logic much cleaner and safer for production workloads.**


3. Optimize queries — indexing, partition pruning, and join strategies

### Optimize Queries in PySpark — Interview Answer

For query optimization in PySpark, I mainly focus on **indexing alternatives, partition pruning, and efficient join strategies**.

#### 1. Indexing

Spark doesn't use traditional database indexes like B-tree indexes. Instead, performance can be improved using:

* **Data skipping** with Delta Lake
* **Z-Ordering** on frequently filtered columns
* Proper **partitioning**
* File statistics and optimized file layouts

```python
df = spark.read.format("delta").load("/data/sales")

df = df.filter("customer_id = 1001")
```

For Delta tables, clustering/Z-Ordering can improve data skipping for frequently queried columns.

#### 2. Partition Pruning

Partition pruning ensures Spark reads **only the required partitions** instead of scanning the entire dataset.

```python
df = spark.read.parquet("/data/sales")

result = df.filter(
    (df.year == 2026) & (df.month == 8)
)
```

If the data is partitioned by `year` and `month`, Spark can skip irrelevant partitions.

**Best practice:** Always filter on partition columns when possible.

#### 3. Join Optimization

Choose the join strategy based on data size.

**Broadcast Join** — for a small table:

```python
from pyspark.sql.functions import broadcast

result = large_df.join(
    broadcast(small_df),
    "customer_id",
    "inner"
)
```

This avoids expensive shuffling of the large dataset.

Other strategies include:

* **Sort-Merge Join** — suitable for large tables
* **Broadcast Hash Join** — suitable when one side is small
* **Shuffle Hash Join** — useful in specific scenarios

### 🎯 Short Interview Answer

> "I optimize PySpark queries by reducing the amount of data scanned and shuffled. I use Delta Lake data skipping/Z-Ordering instead of traditional indexing, apply partition pruning by filtering on partition columns, and choose efficient join strategies such as broadcast joins for small tables and sort-merge joins for large datasets. I also use `explain()` to analyze the execution plan and identify unnecessary scans, shuffles, and joins."


4. Write queries for data validation & reconciliation (source vs target)

For **Data Engineering interviews**, you can demonstrate source-vs-target validation using SQL queries like these.

### 1. Row Count Validation

```sql
-- Source
SELECT COUNT(*) AS source_count
FROM source_table;

-- Target
SELECT COUNT(*) AS target_count
FROM target_table;
```

**Reconciliation:**

```sql
SELECT
    (SELECT COUNT(*) FROM source_table) AS source_count,
    (SELECT COUNT(*) FROM target_table) AS target_count,
    CASE
        WHEN (SELECT COUNT(*) FROM source_table)
           = (SELECT COUNT(*) FROM target_table)
        THEN 'PASS'
        ELSE 'FAIL'
    END AS validation_status;
```

---

### 2. Find Records Missing in Target

```sql
SELECT s.*
FROM source_table s
LEFT JOIN target_table t
    ON s.id = t.id
WHERE t.id IS NULL;
```

This identifies records present in **source but missing in target**.

---

### 3. Find Extra Records in Target

```sql
SELECT t.*
FROM target_table t
LEFT JOIN source_table s
    ON t.id = s.id
WHERE s.id IS NULL;
```

---

### 4. Compare Column Values

```sql
SELECT
    s.id,
    s.name AS source_name,
    t.name AS target_name,
    s.salary AS source_salary,
    t.salary AS target_salary
FROM source_table s
JOIN target_table t
    ON s.id = t.id
WHERE
    COALESCE(s.name, '') <> COALESCE(t.name, '')
    OR COALESCE(s.salary, 0) <> COALESCE(t.salary, 0);
```

---

### 5. NULL Validation

```sql
SELECT
    COUNT(*) AS total_records,
    SUM(CASE WHEN customer_id IS NULL THEN 1 ELSE 0 END) AS null_customer_id,
    SUM(CASE WHEN email IS NULL THEN 1 ELSE 0 END) AS null_email
FROM target_table;
```

---

### 6. Duplicate Validation

```sql
SELECT
    customer_id,
    COUNT(*) AS record_count
FROM target_table
GROUP BY customer_id
HAVING COUNT(*) > 1;
```

---

### 7. Aggregate Reconciliation

Useful for validating financial or transactional data.

```sql
SELECT
    SUM(s.amount) AS source_amount,
    SUM(t.amount) AS target_amount,
    SUM(s.amount) - SUM(t.amount) AS difference
FROM source_table s
CROSS JOIN target_table t;
```

For large datasets, compare aggregates separately:

```sql
SELECT
    (SELECT SUM(amount) FROM source_table) AS source_amount,
    (SELECT SUM(amount) FROM target_table) AS target_amount;
```

---

### 8. Hash-Based Reconciliation

Useful when comparing many columns.

```sql
SELECT
    s.id,
    MD5(CONCAT_WS('|',
        s.name,
        s.email,
        CAST(s.salary AS STRING)
    )) AS source_hash,
    MD5(CONCAT_WS('|',
        t.name,
        t.email,
        CAST(t.salary AS STRING)
    )) AS target_hash
FROM source_table s
JOIN target_table t
    ON s.id = t.id;
```

Then identify mismatches:

```sql
SELECT *
FROM reconciliation
WHERE source_hash <> target_hash;
```

### 🎯 Interview Answer

> **"For source-to-target reconciliation, I validate row counts, duplicate records, NULL values, missing or extra records, column-level differences, and aggregate totals. For large datasets, I also use hash-based reconciliation to compare multiple columns efficiently. I usually perform these checks at both overall and partition/date levels to quickly identify where the mismatch occurred."**


5. Scenario: Detect anomalies in transaction data using SQL

### Scenario: Detect Anomalies in Transaction Data Using SQL

Assume a table:

```sql
transactions (
    transaction_id,
    customer_id,
    transaction_date,
    amount,
    merchant_id,
    status
)
```

### 1. Detect Unusually Large Transactions

Find transactions significantly higher than the customer's average.

```sql
WITH customer_stats AS (
    SELECT
        customer_id,
        AVG(amount) AS avg_amount,
        STDDEV(amount) AS std_amount
    FROM transactions
    GROUP BY customer_id
)
SELECT
    t.*,
    cs.avg_amount,
    cs.std_amount
FROM transactions t
JOIN customer_stats cs
    ON t.customer_id = cs.customer_id
WHERE t.amount > cs.avg_amount + 3 * cs.std_amount;
```

**Logic:** Transactions more than **3 standard deviations** above the customer's normal behavior are flagged.

---

### 2. Detect Duplicate Transactions

```sql
SELECT
    customer_id,
    transaction_date,
    amount,
    merchant_id,
    COUNT(*) AS transaction_count
FROM transactions
GROUP BY
    customer_id,
    transaction_date,
    amount,
    merchant_id
HAVING COUNT(*) > 1;
```

---

### 3. Detect Multiple Transactions Within a Short Time

For example, more than 3 transactions within 10 minutes:

```sql
SELECT *
FROM (
    SELECT
        t.*,
        COUNT(*) OVER (
            PARTITION BY customer_id
            ORDER BY transaction_date
            RANGE BETWEEN INTERVAL 10 MINUTES PRECEDING
                      AND CURRENT ROW
        ) AS txn_count
    FROM transactions t
) x
WHERE txn_count > 3;
```

---

### 4. Detect Sudden Increase in Transaction Amount

Compare the current transaction with the previous transaction.

```sql
WITH txn AS (
    SELECT
        *,
        LAG(amount) OVER (
            PARTITION BY customer_id
            ORDER BY transaction_date
        ) AS previous_amount
    FROM transactions
)
SELECT *
FROM txn
WHERE amount > previous_amount * 5;
```

This identifies transactions **5× larger** than the customer's previous transaction.

---

### 5. Detect Failed Transactions Followed by Success

```sql
WITH txn AS (
    SELECT
        *,
        LAG(status) OVER (
            PARTITION BY customer_id
            ORDER BY transaction_date
        ) AS previous_status
    FROM transactions
)
SELECT *
FROM txn
WHERE previous_status = 'FAILED'
  AND status = 'SUCCESS';
```

### 🎯 Interview Answer

> **"I detect transaction anomalies by establishing normal customer behavior and then identifying deviations. I use window functions like `LAG()` to detect sudden changes, aggregate functions and standard deviation to identify unusually large transactions, and `GROUP BY` to detect duplicates or excessive transaction frequency. For production systems, I would also apply these checks at customer, merchant, and time-window levels."**


## Round 2 – Spark & Big Data

1️⃣ Explain **Spark execution plan (DAG, stages, tasks)**

2️⃣ Difference between **repartition vs coalesce** with use cases

3️⃣ How would you handle **data skew & shuffle optimization?**

4️⃣ **Broadcast join vs sort merge join** — when to use what?

5️⃣ **Scenario:** Spark job running **5+ hours** — how will you optimize it?






##1. Spark Execution Plan: DAG → Stages → Tasks

When a Spark application runs, Spark converts the code into an **execution plan** and breaks the work into **jobs, stages, and tasks**.

### 1. DAG — Directed Acyclic Graph

A **DAG** represents the sequence of transformations Spark needs to perform.

For example:

```python
df = spark.read.parquet("/data/sales")

result = (
    df.filter(df.amount > 1000)
      .groupBy("customer_id")
      .sum("amount")
)
```

Spark creates a DAG similar to:

```text
Read Data
   ↓
Filter
   ↓
Shuffle
   ↓
GroupBy
   ↓
Aggregation
```

Spark optimizes this plan before execution.

---

### 2. Stages

A **stage** is a group of operations that can be executed without requiring a shuffle.

Spark generally creates a **stage boundary when a shuffle occurs**.

Example:

```text
Stage 1
Read → Filter
        ↓
     Shuffle
        ↓
Stage 2
GroupBy → Aggregate
```

Common operations causing shuffle:

* `groupBy()`
* `join()`
* `distinct()`
* `repartition()`
* `orderBy()`

---

### 3. Tasks

A **task** is the smallest unit of execution sent to a Spark executor.

Usually:

> **One task processes one partition of data for a stage.**

For example, if Stage 1 has 100 partitions:

```text
Stage 1
 ├── Task 1  → Partition 1
 ├── Task 2  → Partition 2
 ├── Task 3  → Partition 3
 ...
 └── Task 100 → Partition 100
```

These tasks can run in parallel across executors.

---

### Complete Flow

```text
Spark Application
       ↓
      Job
       ↓
      DAG
       ↓
   ┌─────────┐
   │ Stage 1 │
   └─────────┘
       ↓
    Shuffle
       ↓
   ┌─────────┐
   │ Stage 2 │
   └─────────┘
       ↓
    Tasks
       ↓
   Executors
```

### 🔥 Interview Answer

> **"Spark creates a DAG from the transformations in the application. The DAG is divided into stages based on shuffle boundaries. Each stage is further divided into tasks, where typically one task processes one partition. Tasks are distributed across executors and run in parallel. Narrow transformations stay within the same stage, while wide transformations such as groupBy and join usually create a shuffle and a new stage."**



##2. Repartition vs Coalesce in PySpark

Both are used to **change the number of partitions**, but they behave differently.

| Feature             | `repartition()`           | `coalesce()`                  |
| ------------------- | ------------------------- | ----------------------------- |
| Shuffle             | **Yes**                   | **No** (normally)             |
| Increase partitions | ✅ Yes                     | ❌ No                          |
| Decrease partitions | ✅ Yes                     | ✅ Yes                         |
| Data redistribution | Evenly redistributes data | Combines existing partitions  |
| Performance         | More expensive            | Generally faster              |
| Best use            | Need balanced partitions  | Reduce partitions efficiently |





# # 3. Handling Data Skew & Shuffle Optimization in PySpark

### 1. Identify Data Skew

First, check whether some keys contain significantly more records than others.

```python
from pyspark.sql.functions import count

df.groupBy("customer_id") \
  .agg(count("*").alias("record_count")) \
  .orderBy("record_count", ascending=False) \
  .show(10)
```

If one `customer_id` has millions of records while others have only thousands, it can cause **data skew**.

---

### 2. Use Salting

For a highly skewed join key, add a random salt to distribute the data across partitions.

```python
from pyspark.sql.functions import rand, floor

large_df = large_df.withColumn(
    "salt",
    floor(rand() * 10)
)
```

For the smaller DataFrame, create all possible salt values and join using both keys.

```python
from pyspark.sql.functions import explode, sequence, lit

small_df = small_df.withColumn(
    "salt",
    explode(sequence(lit(0), lit(9)))
)

result = large_df.join(
    small_df,
    ["customer_id", "salt"],
    "inner"
)
```

This distributes a heavily skewed key across multiple partitions.

---

### 3. Broadcast Small Tables

If one side of the join is small enough to fit in executor memory, use a **broadcast join**.

```python
from pyspark.sql.functions import broadcast

result = large_df.join(
    broadcast(small_df),
    "customer_id",
    "inner"
)
```

This avoids shuffling the large DataFrame.

---

### 4. Optimize Shuffle Partitions

Spark's shuffle partition count should match the workload.

```python
spark.conf.set("spark.sql.shuffle.partitions", 400)
```

Too few partitions → large tasks and poor parallelism.

Too many partitions → scheduling overhead and potentially many small tasks/files.

The optimal value depends on **data size, cluster resources, and workload**.

---

### 5. Avoid Unnecessary Shuffles

Operations such as these can trigger expensive shuffles:

```python
groupBy()
join()
distinct()
orderBy()
repartition()
```

For example, if you only need to reduce partitions before writing:

```python
df.coalesce(10).write.parquet("/output")
```

rather than unnecessarily doing:

```python
df.repartition(10)
```

---

### 6. Handle AQE and Skew Join

In modern Spark versions, **Adaptive Query Execution (AQE)** can dynamically optimize shuffle operations.

```python
spark.conf.set(
    "spark.sql.adaptive.enabled", "true"
)

spark.conf.set(
    "spark.sql.adaptive.skewJoin.enabled", "true"
)
```

AQE can detect skewed shuffle partitions and split them into smaller partitions during execution.

---

### 🔥 Interview Answer

> **"I first identify skew by analyzing the distribution of join or group-by keys. For severe skew, I use techniques such as salting or broadcasting a small table. I minimize unnecessary shuffle operations, tune `spark.sql.shuffle.partitions`, and use AQE with skew join optimization. I also monitor the Spark UI to identify stages with long-running tasks, uneven partition sizes, and excessive shuffle read/write. The goal is to distribute data evenly and reduce expensive network and disk I/O."**


##4. Broadcast Join vs Sort-Merge Join in PySpark

The main difference is **how Spark moves and processes the data during the join**.

| Feature     | Broadcast Join                          | Sort-Merge Join                  |
| ----------- | --------------------------------------- | -------------------------------- |
| Best for    | Small + large table                     | Large + large tables             |
| Shuffle     | Avoids shuffle of large table           | Requires shuffle                 |
| Sorting     | No major sorting required               | Both sides are sorted            |
| Memory      | Small table must fit in executor memory | More scalable for large datasets |
| Performance | Very fast when applicable               | Good for large datasets          |
| Typical use | Fact + small dimension                  | Fact + fact / large dimension    |

### 1. Broadcast Join

If one table is small, Spark can send a copy of it to each executor.

```python
from pyspark.sql.functions import broadcast

result = large_df.join(
    broadcast(small_df),
    "customer_id",
    "inner"
)
```

Instead of shuffling the large table:

```text
Large Table ───────────────→ Executors
Small Table → Broadcast → Each Executor
```

### When to use?

Use it when:

* One side of the join is relatively small.
* The small table can comfortably fit in executor memory.
* You want to avoid an expensive shuffle.

**Example:** Joining a 1 TB transaction table with a 50 MB customer dimension table.

---

## 2. Sort-Merge Join

For large tables, Spark generally uses a **Sort-Merge Join**.

Conceptually:

```text
Large Table A
     ↓
 Shuffle
     ↓
 Sort
     ↓
     ┐
     ├── Merge → Join Result
     ┘
 Sort
     ↑
 Shuffle
     ↑
Large Table B
```

Both datasets are shuffled based on the join key and sorted before being merged.

```python
result = large_df1.join(
    large_df2,
    "customer_id",
    "inner"
)
```

### When to use?

Use it when:

* Both datasets are large.
* Broadcasting either side isn't practical.
* You need a scalable join strategy.

**Example:** Joining a 1 TB transaction table with a 500 GB transaction-history table.

---

### ⚠️ Important Interview Point

Don't blindly broadcast a table just because it's smaller.

If the broadcast table is too large for executor memory, it can cause **memory pressure or executor failures**.

You can control the automatic broadcast threshold:

```python
spark.conf.set(
    "spark.sql.autoBroadcastJoinThreshold",
    50 * 1024 * 1024
)
```

Here, Spark can automatically consider broadcasting tables up to approximately **50 MB**.

---

### 🔥 Interview Answer

> **"I use a broadcast join when one side of the join is small enough to fit comfortably in executor memory because it avoids shuffling the large table and is usually much faster. For large-to-large joins, I prefer sort-merge join because it scales better. Before choosing the strategy, I check table sizes, data distribution, join-key skew, and the execution plan using `explain()`."**

**Rule of thumb:**
**Small + Large → Broadcast Join**
**Large + Large → Sort-Merge Join**


##5. Scenario: Spark Job Running for 5+ Hours — How Would You Optimize It?

I would **not immediately increase the cluster size**. First, I would identify the bottleneck using the **Spark UI and execution plan**, then optimize accordingly.

### 1. Check Spark UI First

I would look for:

* Long-running stages
* Tasks with highly unequal execution times
* **Shuffle Read/Write**
* Spill to disk/memory
* Executor GC time
* Failed/retried tasks
* Number and size of partitions

```python
df.explain("formatted")
```

This helps identify expensive scans, joins, exchanges, and sorts.

---

### 2. Check for Data Skew

If one or a few tasks run much longer than the others, I would suspect **data skew**.

```python
df.groupBy("customer_id") \
  .count() \
  .orderBy("count", ascending=False) \
  .show(10)
```

Possible solutions:

* **Salting** skewed keys
* Broadcast the small side of a join
* Enable AQE skew handling

```python
spark.conf.set("spark.sql.adaptive.enabled", "true")
spark.conf.set("spark.sql.adaptive.skewJoin.enabled", "true")
```

---

### 3. Optimize Joins

If I'm joining a huge table with a small dimension table:

```python
from pyspark.sql.functions import broadcast

result = fact_df.join(
    broadcast(dim_df),
    "customer_id"
)
```

For large-to-large joins, I'd investigate the **sort-merge join**, partitioning, skew, and shuffle volume.

---

### 4. Reduce Shuffle

Expensive operations include:

```python
groupBy()
join()
distinct()
orderBy()
repartition()
```

I'd avoid unnecessary `repartition()` and use `coalesce()` when I only need to reduce partitions.

---

### 5. Tune Partitions

Too few partitions can create huge tasks:

```python
spark.conf.set("spark.sql.shuffle.partitions", 400)
```

But I wouldn't blindly set `400`. I'd determine the appropriate value based on **data volume, partition sizes, and available executor cores**.

---

### 6. Filter Data Early

Push filters as close to the source as possible.

Instead of:

```python
df = spark.read.parquet("/sales")
df.groupBy("customer_id").count()
```

Use:

```python
df = (
    spark.read.parquet("/sales")
    .filter("year = 2026 AND month = 8")
)

result = df.groupBy("customer_id").count()
```

This reduces the amount of data Spark needs to scan and process.

---

### 7. Avoid `collect()` and Large Driver Operations

I would check for operations such as:

```python
df.collect()
df.toPandas()
```

These can move huge amounts of data to the driver and cause memory problems.

---

### 8. Optimize File Layout

If reading millions of small files, I'd address the **small-file problem**.

For Delta/Parquet workloads, I would consider:

* Compaction
* Appropriate partitioning
* Data skipping
* Clustering/Z-Ordering where appropriate
* Avoiding over-partitioning

---

### 9. Cache Only When Needed

If the same expensive DataFrame is reused multiple times:

```python
df.cache()
df.count()   # materialize cache
```

But I wouldn't cache everything because unnecessary caching can cause **memory pressure and eviction**.

---

### 10. Check Cluster Resources

Finally, I'd verify:

* Executor CPU utilization
* Executor memory
* GC time
* Number of cores
* Executor count
* Disk/network I/O

Only after optimizing the application would I consider **scaling the cluster**.

### 🔥 Strong Interview Answer

> **"If a Spark job is taking 5+ hours, I first analyze the Spark UI and execution plan rather than immediately increasing the cluster size. I identify slow stages, excessive shuffle, data skew, spills, and uneven task execution. Then I optimize joins using broadcast joins where appropriate, handle skew using salting or AQE, tune shuffle partitions, push filters early for partition pruning, eliminate unnecessary shuffles, optimize file sizes, and cache only reused datasets. Finally, I review executor CPU, memory, GC, and I/O utilization and scale the cluster if the workload is genuinely resource-bound."**

**My optimization order:**
**Spark UI → Skew → Shuffle → Joins → Partitioning → File layout → Memory → Cluster sizing**
